In [1]:
# Cell 1 - Environment setup and raw data ingestion
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = Path("cervical-cancer_csv.csv")

# ponytail: native pandas na_values handles ? sentinels; upgrade to custom coercer if mixed formats emerge
df_raw = pd.read_csv(DATA_PATH, na_values="?")

target_col = "Biopsy"
target_counts = df_raw[target_col].value_counts().to_dict()
missing_summary = df_raw.isnull().sum()
cols_with_na = missing_summary[missing_summary > 0]

print(f"rows={df_raw.shape[0]} | cols={df_raw.shape[1]}")
print(f"target={target_col} | counts={target_counts} | pos_rate={df_raw[target_col].mean():.4f}")
print(f"cols_with_missing={len(cols_with_na)}")
print(f"top_missing={cols_with_na.sort_values(ascending=False).head(5).to_dict()}")


In [2]:
# Cell 2 - Feature triage and leak-free stratified split setup
from sklearn.model_selection import StratifiedKFold

# ponytail: drop diagnostic target leaks and sparse >90% nulls directly; upgrade to indicator flags if signal needed
diagnostic_leakage = ["Hinselmann", "Schiller", "Citology"]
sparse_cols = ["STDs: Time since first diagnosis", "STDs: Time since last diagnosis"]
drop_cols = diagnostic_leakage + sparse_cols + [target_col]

X = df_raw.drop(columns=drop_cols)
y = df_raw[target_col].astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

print(f"features_retained={X.shape[1]} | total_samples={len(X)}")
for f, (tr_idx, va_idx) in enumerate(folds):
    tr_pos = int(y.iloc[tr_idx].sum())
    va_pos = int(y.iloc[va_idx].sum())
    print(f"fold={f+1}/5 | train_len={len(tr_idx)} pos={tr_pos} | val_len={len(va_idx)} pos={va_pos}")


In [3]:
# Cell 3 - In-depth data profiling and zero-variance feature elimination
import numpy as np
import pandas as pd

# Check zero-variance / constant features
constant_features = [col for col in X.columns if X[col].dropna().nunique() <= 1]

# Drop zero-variance columns (e.g. STDs:cervical condylomatosis, STDs:AIDS)
X_clean = X.drop(columns=constant_features)

# Missing rate profile on cleaned features
missing_rates = (X_clean.isnull().mean() * 100).sort_values(ascending=False)
active_missing = missing_rates[missing_rates > 0]

# Compute preliminary univariate correlation with target
univariate_corr = X_clean.corrwith(y).sort_values(ascending=False)

print(f"constant_features_dropped={constant_features}")
print(f"remaining_features={X_clean.shape[1]}")
print(f"top_missing_rates={active_missing.head(5).round(2).to_dict()}")
print(f"top_positive_risk_factors={univariate_corr.head(5).round(4).to_dict()}")
print(f"top_negative_risk_factors={univariate_corr.dropna().tail(3).round(4).to_dict()}")


In [4]:
# Cell 4 - Ablation Study: Family 1 (Linear and Regularized Models)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

family_1_models = {
    "LR_L2_Unweighted": lambda: LogisticRegression(
        C=1.0, random_state=SEED, max_iter=1000
    ),
    "LR_L2_Balanced": lambda: LogisticRegression(
        C=1.0, class_weight="balanced", random_state=SEED, max_iter=1000
    ),
    "LR_L1_Lasso_Balanced": lambda: LogisticRegression(
        solver="saga", l1_ratio=1.0, C=0.1, class_weight="balanced", random_state=SEED, max_iter=2000
    ),
    "LR_ElasticNet_Balanced": lambda: LogisticRegression(
        solver="saga", l1_ratio=0.5, C=0.1, class_weight="balanced", random_state=SEED, max_iter=2000
    ),
}

f1_oof_preds = {m: np.zeros(len(y)) for m in family_1_models}
f1_nonzero_counts = {m: [] for m in family_1_models}

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr_raw, y_tr = X_clean.iloc[tr_idx], y.iloc[tr_idx]
    X_va_raw, y_va = X_clean.iloc[va_idx], y.iloc[va_idx]

    # ponytail: fold-isolated median imputation + z-score scaling; strict zero-leakage contract
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(imputer.fit_transform(X_tr_raw))
    X_va = scaler.transform(imputer.transform(X_va_raw))

    for name, model_fn in family_1_models.items():
        clf = model_fn()
        clf.fit(X_tr, y_tr)
        f1_oof_preds[name][va_idx] = clf.predict_proba(X_va)[:, 1]
        f1_nonzero_counts[name].append(int(np.sum(clf.coef_ != 0)))

random_baseline = float(y.mean())
print(f"random_baseline_prauc={random_baseline:.4f}")

for name, preds in f1_oof_preds.items():
    pr = average_precision_score(y, preds)
    roc = roc_auc_score(y, preds)
    brier = brier_score_loss(y, preds)
    avg_nz = np.mean(f1_nonzero_counts[name])
    print(f"model={name:23s} | pr_auc={pr:.4f} | roc_auc={roc:.4f} | brier={brier:.4f} | avg_active_features={avg_nz:.1f}")


In [5]:
# Cell 5 - Ablation Study: Family 2 (Bagging and Decision Forests)
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

family_2_models = {
    "RF_Unweighted_Deep": lambda: RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=SEED
    ),
    "RF_Balanced_Deep": lambda: RandomForestClassifier(
        n_estimators=100, max_depth=None, class_weight="balanced", random_state=SEED
    ),
    "RF_Balanced_Shallow_d3": lambda: RandomForestClassifier(
        n_estimators=100, max_depth=3, class_weight="balanced", random_state=SEED
    ),
    "ExtraTrees_Balanced_d3": lambda: ExtraTreesClassifier(
        n_estimators=100, max_depth=3, class_weight="balanced", random_state=SEED
    ),
}

f2_oof_preds = {m: np.zeros(len(y)) for m in family_2_models}

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr_raw, y_tr = X_clean.iloc[tr_idx], y.iloc[tr_idx]
    X_va_raw, y_va = X_clean.iloc[va_idx], y.iloc[va_idx]

    # ponytail: fold-isolated median imputation for tree models; trees invariant to monotonic scaling
    imputer = SimpleImputer(strategy="median")
    X_tr = imputer.fit_transform(X_tr_raw)
    X_va = imputer.transform(X_va_raw)

    for name, model_fn in family_2_models.items():
        clf = model_fn()
        clf.fit(X_tr, y_tr)
        f2_oof_preds[name][va_idx] = clf.predict_proba(X_va)[:, 1]

print(f"random_baseline_prauc={random_baseline:.4f}")

for name, preds in f2_oof_preds.items():
    pr = average_precision_score(y, preds)
    roc = roc_auc_score(y, preds)
    brier = brier_score_loss(y, preds)
    print(f"model={name:23s} | pr_auc={pr:.4f} | roc_auc={roc:.4f} | brier={brier:.4f}")
